# Fixed-fold LR multistart in softplus coordinates (Kaggle)

Same three-zone, fixed-fold fitting protocol as the log-coordinate notebook, now using
\[C=C_*\,\operatorname{softplus}(u),\qquad \operatorname{softplus}(u)=\log(1+e^u).\]
This guarantees positive calcium for finite \(u\). The scale \(C_*\) is explicit because it controls the curvature of the transformation; the default is 0.05 µM.

The previous selected models are reused for **geometry**: equilibrium anchors, physical calcium thresholds, fixed fold input, and the old fold flight times initialize the search. Their log-coordinate coefficients are not copied: changing coordinates changes the vector field. The first setup cell refits the LR field in softplus coordinates and solves a new neutral-fold start, then the batch gathers 32 distinct valid starts per AM/AFM/FM and optimizes them.

Use a CPU Kaggle session with Internet enabled. Results are saved under `/kaggle/working/pwl_runs` and can be downloaded as one ZIP. This is a finite search: 32 accepted fold starts do not guarantee 32 completed cycle fits.

In [ ]:
import os, sys, subprocess, pathlib, json, time
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["OMP_NUM_THREADS"]="1"
os.environ["MKL_NUM_THREADS"]="1"
AVAILABLE_CPUS=len(os.sched_getaffinity(0)) if hasattr(os,"sched_getaffinity") else (os.cpu_count() or 1)
print("Visible CPU cores:", AVAILABLE_CPUS)

In [ ]:
REPO_URL="https://github.com/KirylMP/PWL-NEST.git"
REPO_BRANCH="experiment/lr-softplus-fixedfold-kaggle-20261116"
REPO_DIR=pathlib.Path("/kaggle/working/PWL-NEST")
if not (REPO_DIR/".git").exists():
    subprocess.run(["git","clone","--depth","1","--filter=blob:none","--branch",REPO_BRANCH,"--single-branch",REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(["git","-C",str(REPO_DIR),"fetch","--depth","1","origin",REPO_BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO_DIR),"checkout","-B",REPO_BRANCH,"FETCH_HEAD"],check=True)
subprocess.run(["git","-C",str(REPO_DIR),"sparse-checkout","set","scripts","results/coordinate_experiment","results/fold_anchors","paper/data/random_selected"],check=True)
required=["scripts/prepare_softplus_fold_anchors.py","scripts/run_transformed_server_batch.py","scripts/summarize_transformed_server_batch.py","scripts/optimize_fixed_fold_cycles.py","scripts/verify_transformed_fold.py"]
missing=[x for x in required if not (REPO_DIR/x).exists()]
if missing: raise FileNotFoundError(missing)
print("Pinned revision:",subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],text=True).strip())

In [ ]:
try:
    import numpy as np
    import scipy
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","numpy","scipy"],check=True)
    import numpy as np
    import scipy
print("NumPy",np.__version__,"SciPy",scipy.__version__)

## Prepare the softplus starting folds

This reuses the previous fixed points and fold geometry, but fits a fresh field in the new coordinate. It writes one checked neutral-fold anchor per model. If a model cannot satisfy the fold and itinerary checks, stop here and inspect its attempt JSON; do not launch the multistart batch with a missing anchor.

In [ ]:
REPO_DIR=pathlib.Path("/kaggle/working/PWL-NEST")
SOFTPLUS_SCALE_UM=0.05
FOLD_START_NFEV=500
prep=[sys.executable,"-u","scripts/prepare_softplus_fold_anchors.py",
      "--cases","AM","AFM","FM","--scale",str(SOFTPLUS_SCALE_UM),
      "--fold-nfev",str(FOLD_START_NFEV)]
subprocess.run(prep,cwd=REPO_DIR,check=True)
for case in ("AM","AFM","FM"):
    anchor=REPO_DIR/"results"/"fold_anchors"/case/"softplus"/"seed_0.json"
    if not anchor.is_file(): raise FileNotFoundError(anchor)
    d=json.loads(anchor.read_text())
    print(case,"fold I",d["I_fold"],"matching",d["matching_max"],"logM",d["logM"],"thresholds u",d["cuts"])

## Smoke pass

One candidate and one fit evaluation per model check the wiring only; it is not a quality or convergence test.

In [ ]:
RUNS_DIR=pathlib.Path("/kaggle/working/pwl_runs")
RUNS_DIR.mkdir(parents=True,exist_ok=True)
SMOKE_DIR=RUNS_DIR/"transformed_softplus_smoke"
cmd=[sys.executable,"-u","scripts/run_transformed_server_batch.py","--output",str(SMOKE_DIR),
     "--combos","AM:softplus","AFM:softplus","FM:softplus","--valid-per-combo","1",
     "--workers",str(min(3,AVAILABLE_CPUS)),"--fold-nfev","100","--fit-nfev","1",
     "--max-attempts","8","--timeout","900"]
if (SMOKE_DIR/"batch.json").exists(): cmd.append("--resume")
subprocess.run(cmd,cwd=REPO_DIR,check=True)
subprocess.run([sys.executable,"scripts/summarize_transformed_server_batch.py",str(SMOKE_DIR)],cwd=REPO_DIR,check=True)

## Configure and launch

Default: 32 distinct fold starts per model. Keep worker count modest on Kaggle; each worker integrates cycles and does nonlinear optimization. Re-running the launch cell resumes the same manifest. Do not change settings when resuming that batch.

In [ ]:
VALID_PER_MODEL=32
WORKERS=min(4,AVAILABLE_CPUS)
FOLD_NFEV=1200
FIT_NFEV=80
MAX_ATTEMPTS=5000
TIMEOUT=3600
BATCH_NAME=f"transformed_softplus_Cstar{int(round(SOFTPLUS_SCALE_UM*1000)):03d}_fold32"
BATCH_DIR=RUNS_DIR/BATCH_NAME
LOG_DIR=RUNS_DIR/"logs"
LOG_DIR.mkdir(parents=True,exist_ok=True)
BATCH_LOG=LOG_DIR/f"{BATCH_NAME}.log"
PID_FILE=LOG_DIR/f"{BATCH_NAME}.pid"
print("Models AM/AFM/FM; accepted-fold target:",VALID_PER_MODEL*3,"workers:",WORKERS,"C*:",SOFTPLUS_SCALE_UM,"uM")

In [ ]:
cmd=[sys.executable,"-u","scripts/run_transformed_server_batch.py","--output",str(BATCH_DIR),
     "--combos","AM:softplus","AFM:softplus","FM:softplus",
     "--valid-per-combo",str(VALID_PER_MODEL),"--workers",str(WORKERS),
     "--fold-nfev",str(FOLD_NFEV),"--fit-nfev",str(FIT_NFEV),
     "--max-attempts",str(MAX_ATTEMPTS),"--timeout",str(TIMEOUT)]
if (BATCH_DIR/"batch.json").exists(): cmd.append("--resume")
if PID_FILE.exists():
    old=PID_FILE.read_text().strip()
    active=subprocess.run(["ps","-p",old,"-o","args="],text=True,capture_output=True).stdout
    if "run_transformed_server_batch.py" in active:
        raise RuntimeError("Batch appears active; monitor it instead of launching a duplicate.")
with BATCH_LOG.open("a",buffering=1) as log:
    proc=subprocess.Popen(cmd,cwd=REPO_DIR,stdout=log,stderr=subprocess.STDOUT,start_new_session=True,close_fds=True)
PID_FILE.write_text(str(proc.pid)+"\n")
print("Started PID",proc.pid,"log",BATCH_LOG)

## Monitor

Run this cell repeatedly while Kaggle is active. Setup and anchor generation happen before the batch and are not included in the candidate summary.

In [ ]:
subprocess.run(["tail","-n","40",str(BATCH_LOG)],check=False)
subprocess.run([sys.executable,"scripts/summarize_transformed_server_batch.py",str(BATCH_DIR)],cwd=REPO_DIR,check=False)
if PID_FILE.exists():
    subprocess.run(["ps","-p",PID_FILE.read_text().strip(),"-o","pid,etime,time,%cpu,stat,args"],check=False)

## Package results

Run after the batch has ended. The archive includes accepted fold starts, failed attempts, fitted candidates, verification records, summaries, and logs.

In [ ]:
import shutil
from IPython.display import FileLink, display
RESULT_ZIP=pathlib.Path("/kaggle/working")/f"{BATCH_NAME}.zip"
if RESULT_ZIP.exists(): RESULT_ZIP.unlink()
shutil.make_archive(str(RESULT_ZIP.with_suffix("")),"zip",root_dir=BATCH_DIR)
print("ZIP size MiB:",round(RESULT_ZIP.stat().st_size/(1024**2),2))
display(FileLink(str(RESULT_ZIP)))